# Analyze locomotion during recordings with stim
The abf files are supposed to have 3 channels (sweep 0): LFP, rotary encoder signal, stimulation analog signal

In [ ]:
import sys
import os
notebook_dir = os.getcwd()
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))  # Add the project root directory to the path
import numpy as np
import matplotlib.pyplot as plt
import custom_io as cio
from locomotion_functions import *
import pandas as pd
from datetime import datetime
import warnings
import seaborn as sns
from scipy.io import savemat
from experiment_classes import Experiment, ExperimentMetaData

In [ ]:
show_figs = False
save_results = True

## Load folder containing all loco files to process
Folder should contain "used_files.xlsx" to filter files.

In [ ]:
# Should be SLE/include_in_analysis/
folder = cio.open_dir("Choose folder to load abf files from")

fpath_info = os.path.join(folder, "used_files.xlsx")
assert os.path.exists(fpath_info)
df_info = pd.read_excel(fpath_info)

fpaths = []
for root, folders, files in os.walk(folder):
    for file in files:
        if file.endswith(".abf"):
            fpath = os.path.join(root, file)
            print(fpath)
            assert os.path.exists(fpath)
            fpaths.append(fpath)

            

In [ ]:
if save_results:
    output_folder = cio.open_dir("Choose folder to save results")
    assert os.path.exists(output_folder)

In [ ]:
experiments = []
for fpath in fpaths:
    print(fpath)
    exp = Experiment(fpath)
    experiments.append(exp)

experiments_metadata = []
for i, fpath in enumerate(fpaths):
    print(fpath)
    row = df_info[df_info["file_name"] == os.path.basename(fpath)]
    if len(row) == 0:
        raise FileNotFoundError(f"File {fpath} not found in the info file!")
    row = row.iloc[0]
    uuid = row["uuid"]
    mouse_id = row[ "mouse_id"]
    mouse_type = row["mouse_type"]
    fname = os.path.basename(fpath)
    date = row["date"] 
    exp_type = row["exp_type"]
    comment = row["comment"]
    exp_meta = ExperimentMetaData(uuid, mouse_id, mouse_type, fname, date, exp_type, comment)
    experiments_metadata.append(exp_meta)

In [ ]:
#data_to_export = {
#    "objects": [
#        {
#            "array1": obj.array1,
#            "array2": obj.array2,
#            "int1": obj.int1,
#            "int2": obj.int2
#        }
#        for obj in objects
#    ]
#}

In [ ]:
dict_loco_data = {"uuid": [], "mouse_id": [], "mouse_type": [], "fname": [], "date": [], "exp_type": [], "segment_type": [], "total_distance_absolute": [], "max_speed": [], "running_percent": [], "n_episodes": []}

for i_experiment, experiment in enumerate(experiments):
    if experiment.idx_stim_begin is None:
        print(f"No stim begin/end detected for {experiment.fpath_abf}, skipping...")
    else:
        # add pre segment
        dict_loco_data["uuid"].append(experiments_metadata[i_experiment].uuid)
        dict_loco_data["mouse_id"].append(experiments_metadata[i_experiment].mouse_id)
        dict_loco_data["mouse_type"].append(experiments_metadata[i_experiment].mouse_type)
        dict_loco_data["fname"].append(experiments_metadata[i_experiment].fname)
        dict_loco_data["date"].append(experiments_metadata[i_experiment].date)
        dict_loco_data["exp_type"].append(experiments_metadata[i_experiment].exp_type)
        dict_loco_data["segment_type"].append("pre")
        dict_loco_data["total_distance_absolute"].append(experiment.total_distance_absolute_pre)
        dict_loco_data["max_speed"].append(experiment.max_speed_pre)
        dict_loco_data["running_percent"].append(experiment.running_percent_pre)
        dict_loco_data["n_episodes"].append(experiment.n_episodes_pre)
        # addd post segment
        dict_loco_data["uuid"].append(experiments_metadata[i_experiment].uuid)
        dict_loco_data["mouse_id"].append(experiments_metadata[i_experiment].mouse_id)
        dict_loco_data["mouse_type"].append(experiments_metadata[i_experiment].mouse_type)
        dict_loco_data["fname"].append(experiments_metadata[i_experiment].fname)
        dict_loco_data["date"].append(experiments_metadata[i_experiment].date)
        dict_loco_data["exp_type"].append(experiments_metadata[i_experiment].exp_type)
        dict_loco_data["segment_type"].append("post")
        dict_loco_data["total_distance_absolute"].append(experiment.total_distance_absolute_post)
        dict_loco_data["max_speed"].append(experiment.max_speed_post)
        dict_loco_data["running_percent"].append(experiment.running_percent_post)
        dict_loco_data["n_episodes"].append(experiment.n_episodes_post)

In [ ]:
df = pd.DataFrame(dict_loco_data)

In [ ]:
# sort by mouse_id, then date, then file_name
df = df.sort_values(by=["mouse_type", "exp_type", "mouse_id", "date", "fname"])

In [ ]:
# add proper type of controls
uuids_szsd_ctl = ["097ce8628a724b1fb85981ff3923e693", "35f7374e92c14eb78b7d6acb018748be", "1915f754cd004ecba316841ecc032449", "ed2be94ff1de4af29775863b0ef6038c"]
df["ctl_type"] = ""
# proper sz mimic:
df.loc[~df["uuid"].isin(uuids_szsd_ctl) & (df["exp_type"] == "sz_mimic_ctl"), "ctl_type"] = "Sz mimic ctl"
# szsd ctl:
df.loc[df["uuid"].isin(uuids_szsd_ctl), "ctl_type"] = "Sz + SD ctl"
# sd 20s: transgenic mice
df.loc[(df["mouse_type"] == "tg") & (df["exp_type"] == "sd_ctl"), "ctl_type"] = "SD ctl (20s)"
# sd 4s: aav mice
df.loc[(df["mouse_type"] == "aav") & (df["exp_type"] == "sd_ctl"), "ctl_type"] = "SD ctl (4s)"
# change "exp_type" to "ctl" whenever ctl is in the exp_type
df.loc[df["exp_type"].str.contains("ctl"), "exp_type"] = "ctl"

In [ ]:
# create differences grouped by fname, segment_type=post - pre
df_diff = df.pivot_table(index=["uuid", "mouse_id", "mouse_type", "fname", "date", "exp_type", "ctl_type"], columns="segment_type", values=["total_distance_absolute", "max_speed", "n_episodes", "running_percent"])
# calculate differences
df_diff["total_distance_absolute_diff"] = df_diff["total_distance_absolute"]["post"] - df_diff["total_distance_absolute"]["pre"]
df_diff["max_speed_diff"] = df_diff["max_speed"]["post"] - df_diff["max_speed"]["pre"]
df_diff["n_episodes_diff"] = df_diff["n_episodes"]["post"] - df_diff["n_episodes"]["pre"]
df_diff["running_percent_diff"] = df_diff["running_percent"]["post"] - df_diff["running_percent"]["pre"]
# drop multiindex columns
df_diff.columns = df_diff.columns.droplevel(1)
# drop columns that appear more tha nonce (i.e. the pre/post columns that now have one level)
df_diff = df_diff.drop(columns=["total_distance_absolute", "max_speed", "n_episodes", "running_percent"])
# reset index
df_diff = df_diff.reset_index()

In [ ]:
df_diff = df_diff.sort_values(by=["mouse_type", "exp_type", "mouse_id", "date", "fname"])

In [ ]:
if save_results:
    df.to_excel(f"{output_folder}\\cannula_loco_data.xlsx", index=False)
    df_diff.to_excel(f"{output_folder}\\cannula_loco_data_diff.xlsx", index=False)

In [ ]:
if show_figs:
    fig, axs = plt.subplots(4, 1, figsize=(18, 24))
    sns.boxplot(x="exp_type", y="total_distance_absolute", hue="segment_type", data=df, ax=axs[0])
    axs[0].set_title("Total absolute distance")
    axs[0].set_ylabel("Total absolute distance (cm)")
    sns.boxplot(x="exp_type", y="max_speed", hue="segment_type", data=df, ax=axs[1])
    axs[1].set_title("Max speed")
    axs[1].set_ylabel("Max speed (cm/s)")
    sns.boxplot(x="exp_type", y="running_percent", hue="segment_type", data=df, ax=axs[2])
    axs[2].set_title("Running percent")
    axs[2].set_ylabel("Running percent (%)")
    sns.boxplot(x="exp_type", y="n_episodes", hue="segment_type", data=df, ax=axs[3])
    axs[3].set_title("Number of episodes")
    axs[3].set_ylabel("Number of episodes")
    plt.show()

In [ ]:
if show_figs:
    fig, axs = plt.subplots(4, 1, figsize=(18, 24))
    sns.boxplot(x="exp_type", y="total_distance_absolute_diff", hue="exp_type", data=df_diff, ax=axs[0])
    axs[0].set_title("Total absolute distance")
    axs[0].set_ylabel("Total distance difference (cm)")
    sns.boxplot(x="exp_type", y="max_speed_diff", hue="exp_type", data=df_diff, ax=axs[1])
    axs[1].set_title("Max speed")
    axs[1].set_ylabel("Max speed difference (cm/s)")
    sns.boxplot(x="exp_type", y="n_episodes_diff", hue="exp_type", data=df_diff, ax=axs[2])
    axs[2].set_title("Number of episodes")
    axs[2].set_ylabel("Number of episodes difference")
    sns.boxplot(x="exp_type", y="running_percent_diff", hue="exp_type", data=df_diff, ax=axs[3])
    axs[3].set_title("% of time spent running")
    axs[3].set_ylabel("Running percent difference (%)")
    plt.show()

## Check for one mouse (WEZ8967) where two sz mimic ctl exist: one that is like the sz mimic pattern (frequencies), and another which has different frequencies.

In [ ]:
# check wez8967: compare the 2-2 different control paradigms 
ctl_uuids_1 = ["097ce8628a724b1fb85981ff3923e693", "35f7374e92c14eb78b7d6acb018748be"]
ctl_uuids_2 = ["1915f754cd004ecba316841ecc032449", "ed2be94ff1de4af29775863b0ef6038c"]

df1 = df[df["uuid"].isin(ctl_uuids_1)]
df2 = df[df["uuid"].isin(ctl_uuids_2)]

df1["type"] = "ctl_unlike_sz_mimic"
df2["type"] = "ctl_like_sz_mimic"

df12 = pd.concat([df1, df2])
if show_figs:
    fig, axs = plt.subplots(2, 2, figsize=(8, 8))
    sns.scatterplot(x="segment_type", y="total_distance_absolute", hue="type", data=df12, ax=axs[0][0])
    axs[0][0].set_title("Total absolute distance")
    axs[0][0].set_ylabel("Total absolute distance (cm)")
    sns.scatterplot(x="segment_type", y="max_speed", hue="type", data=df12, ax=axs[1][0])
    axs[1][0].set_title("Max speed")
    axs[1][0].set_ylabel("Max speed (cm/s)")
    sns.scatterplot(x="segment_type", y="running_percent", hue="type", data=df12, ax=axs[0][1])
    axs[0][1].set_title("Running percent")
    axs[0][1].set_ylabel("Running percent (%)")
    sns.scatterplot(x="segment_type", y="n_episodes", hue="type", data=df12, ax=axs[1][1])
    axs[1][1].set_title("Number of episodes")
    axs[1][1].set_ylabel("Number of episodes")
    plt.show()

Conclusion: Can include the "ctl unlike sz mimic pattern" too, as it can only make our results weaker (all metrics increase compared to pre-stim)

# Check pooling of control recordings
Problem: there are 4 types of controls: SD 4s, SD 20s, sz mimic, sz + sd mimic. Also, there are aav and transgenic mice. Check if these two axes of differentiation can be pooled.

In [ ]:
df_diff_proper_sz_ctl = df_diff[df_diff["ctl_type"] == "Sz mimic ctl"]
df_diff_szsd_ctl = df_diff[df_diff["ctl_type"] == "Sz + SD ctl"]
df_diff_sd_4s_ctl = df_diff[df_diff["ctl_type"] == "SD ctl (4s)"]
df_diff_sd_20s_ctl = df_diff[df_diff["ctl_type"] == "SD ctl (20s)"]

## 1. Check proper sz mimic ctl (i.e. exclude sz+sd pattern) aav vs tg mice. The difference should look similar for both groups

In [ ]:
if show_figs:
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))
    sns.violinplot(x="mouse_type", y="total_distance_absolute_diff", hue = "mouse_type", data=df_diff_proper_sz_ctl, ax = axs[0][0], inner="point")
    axs[0][0].set_title("Total absolute distance diff")
    axs[0][0].set_ylabel("Total absolute distance post-pre difference (cm)")
    axs[0][0].set_xlabel("Mouse type")
    sns.violinplot(x="mouse_type", y="max_speed_diff", hue = "mouse_type", data=df_diff_proper_sz_ctl, ax = axs[1][0], inner="point")
    axs[1][0].set_title("Max speed diff")
    axs[1][0].set_ylabel("Max speed post-pre difference (cm/s)")
    axs[1][0].set_xlabel("Mouse type")
    sns.violinplot(x="mouse_type", y="running_percent_diff", hue = "mouse_type", data=df_diff_proper_sz_ctl, ax = axs[0][1], inner="point")
    axs[0][1].set_title("Running percent diff")
    axs[0][1].set_ylabel("Running percent post-pre difference (%)")
    axs[0][1].set_xlabel("Mouse type")
    sns.violinplot(x="mouse_type", y="n_episodes_diff", hue = "mouse_type", data=df_diff_proper_sz_ctl, ax = axs[1][1], inner="point")
    axs[1][1].set_title("Number of episodes diff")
    axs[1][1].set_ylabel("Number of episodes post-pre difference")
    axs[1][1].set_xlabel("Mouse type")
    plt.show()

Conclusion: proper sz mimic recordings from tg and aav mice can be pooled, so we have a single "proper sz mimic" category.

## Compare SD 4s (aav), SD 20s (tg), sz+sd ctl (tg 1 animal), sz mimic ctl (tg+aav)

In [ ]:
# Then, plot 4 categories: Sd 4s, SD 20s, sz+sd ctl, sz mimic ctl deltas to see if they can be pooled.
df_all_ctls = pd.concat([df_diff_szsd_ctl, df_diff_proper_sz_ctl, df_diff_sd_4s_ctl, df_diff_sd_20s_ctl])
if show_figs:
    fig, axs = plt.subplots(2, 2, figsize=(14, 14))
    sns.violinplot(x="ctl_type", y="total_distance_absolute_diff", hue = "ctl_type", data=df_all_ctls, ax = axs[0][0], inner="point")
    axs[0][0].set_title("Total absolute distance diff")
    axs[0][0].set_ylabel("Total absolute distance post-pre difference (cm)")
    axs[0][0].set_xlabel("Ctl type")

    sns.violinplot(x="ctl_type", y="max_speed_diff", hue = "ctl_type", data=df_all_ctls, ax = axs[1][0], inner="point")
    axs[1][0].set_title("Max speed diff")
    axs[1][0].set_ylabel("Max speed post-pre difference (cm/s)")
    axs[1][0].set_xlabel("Ctl type")

    sns.violinplot(x="ctl_type", y="running_percent_diff", hue = "ctl_type", data=df_all_ctls, ax = axs[0][1], inner="point")
    axs[0][1].set_title("Running percent diff")
    axs[0][1].set_ylabel("Running percent post-pre difference (%)")
    axs[0][1].set_xlabel("Ctl type")

    sns.violinplot(x="ctl_type", y="n_episodes_diff", hue = "ctl_type", data=df_all_ctls, ax = axs[1][1], inner="point")
    axs[1][1].set_title("Number of episodes diff")
    axs[1][1].set_ylabel("Number of episodes post-pre difference")
    axs[1][1].set_xlabel("Ctl type")

    plt.show()


# Export to workspace

### Show example plot of the data

In [ ]:
t = experiments[0].t
lfp_y = experiments[0].lfp_y
speed = experiments[0].speed
totdist_abs = experiments[0].total_distance_absolute
stim_y = experiments[0].stim_y
i_bl_begin = experiments[0].idx_pre_begin
i_bl_end = experiments[0].idx_pre_end
i_stim_begin = experiments[0].idx_stim_begin
i_stim_end = experiments[0].idx_stim_end
i_post_begin = experiments[0].idx_post_begin
i_post_end = experiments[0].idx_post_end
n_frames = len(t)

if show_figs:
    fig = plt.figure(figsize=(18, 6))
    ax1 = fig.add_subplot(411)
    ax1.plot(t, lfp_y, label="LFP")
    ax1.set_title("LFP")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("LFP (mV)")
    ax1.legend()
    ax2 = fig.add_subplot(412)
    ax2.plot(t, speed, label="Speed")
    ax2.axvline(x=t[i_bl_begin], color='r', linestyle='--', label="Baseline begin")
    ax2.axvline(x=t[i_bl_end], color='g', linestyle='--', label="Baseline end")
    ax2.axvline(x=t[i_stim_begin], color='b', linestyle='--', label="Stim begin")
    ax2.axvline(x=t[i_stim_end], color='c', linestyle='--', label="Stim end")
    ax2.axvline(x=t[i_post_begin], color='m', linestyle='--', label="Post begin")
    ax2.axvline(x=t[i_post_end], color='y', linestyle='--', label="Post end")
    ax2.set_title("Speed")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Speed (cm/s)")
    ax2.legend()
    ax3 = fig.add_subplot(413)
    ax3.plot(t, stim_y, label="Stim")
    ax3.set_title("Stim")
    ax3.set_xlabel("Time (s)")
    ax3.set_ylabel("Stim (V)")
    ax3.legend()
    ax4 = fig.add_subplot(414)
    ax4.plot(t, totdist_abs, label="Total absolute distance")
    ax4.set_title("Total absolute distance")


    plt.tight_layout()
    plt.show()

In [ ]:
# get minimum value for lfp, speed, stim over all recordings
min_lfp = np.inf
min_speed = np.inf
min_stim = np.inf
min_totdist = np.inf
for experiment in experiments:
    min_lfp = min(min_lfp, np.min(experiment.lfp_y))
    min_speed = min(min_speed, np.min(experiment.speed))
    min_stim = min(min_stim, np.min(experiment.stim_y))
    min_totdist = min(min_totdist, np.min(experiment.total_distance_absolute))
print(f"Minimum LFP: {min_lfp}")
print(f"Minimum speed: {min_speed}")
print(f"Minimum stim: {min_stim}")
print(f"Minimum total distance: {min_totdist}")


In [ ]:
# get all traces and timestamps as a 2d array. Pad the end with -100 to make all traces the same length
n_traces = len(experiments)
uuids  = []
mouse_ids = []
mouse_types = []
exp_types = []
ctl_types = []


idx_pre_begin = np.array([experiment.idx_pre_begin for experiment in experiments])
idx_pre_end = np.array([experiment.idx_pre_end for experiment in experiments])
idx_stim_begin = np.array([experiment.idx_stim_begin for experiment in experiments])
idx_stim_end = np.array([experiment.idx_stim_end for experiment in experiments])
idx_post_begin = np.array([experiment.idx_post_begin for experiment in experiments])
idx_post_end = np.array([experiment.idx_post_end for experiment in experiments])
n_frames = np.array([len(experiment.t[idx_pre_begin[i_exp]:idx_post_end[i_exp]+1]) for i_exp, experiment in enumerate(experiments)])
n_frames_longest = np.max(n_frames)

# pad with values that do not occur in the data
lfp = np.full((n_traces, n_frames_longest), 1.5*min_lfp)  
timestamps = np.full((n_traces, n_frames_longest), np.inf)  # time stamps start with 0
speed_cmps = np.full((n_traces, n_frames_longest), np.inf)  
led_status = np.full((n_traces, n_frames_longest), np.inf)  




for i, experiment in enumerate(experiments):
    # get the lfp, speed and stim traces
    lfp[i, :n_frames[i]] = experiment.lfp_y[idx_pre_begin[i]:idx_post_end[i]+1]
    timestamps[i, :n_frames[i]] = experiment.t[idx_pre_begin[i]:idx_post_end[i]+1] - experiment.t[experiment.idx_stim_begin]  # time stamps start with 0
    speed_cmps[i, :n_frames[i]] = experiment.speed[idx_pre_begin[i]:idx_post_end[i]+1]
    led_status[i, :n_frames[i]] = experiment.stim_y[idx_pre_begin[i]:idx_post_end[i]+1]
    uuids.append(experiments_metadata[i].uuid)
    mouse_ids.append(experiments_metadata[i].mouse_id)
    mouse_types.append(experiments_metadata[i].mouse_type)
    exp_types.append(experiments_metadata[i].exp_type)
    ctl_types.append(df[df["uuid"] == experiments_metadata[i].uuid]["ctl_type"].values[0])
# convert to numpy 
uuids = np.array(uuids)
mouse_ids = np.array(mouse_ids)
mouse_types = np.array(mouse_types)
exp_types = np.array(exp_types)
ctl_types = np.array(ctl_types)

# correct indices due to the cut
idx_pre_end = idx_pre_end - idx_pre_begin  # traces start with pre
idx_stim_begin = idx_stim_begin - idx_pre_begin 
idx_stim_end = idx_stim_end - idx_pre_begin 
idx_post_begin = idx_post_begin - idx_pre_begin 
idx_post_end = idx_post_end - idx_pre_begin 
idx_pre_begin = idx_pre_begin - idx_pre_begin


workspace_dict = {
    "uuids": uuids,
    "mouse_ids": mouse_ids,
    "mouse_types": mouse_types,
    "exp_types": exp_types,
    "ctl_types": ctl_types,
    "lfp": lfp,
    "speed_cmps": speed_cmps,
    "timestamps": timestamps,
    "led_status": led_status,
    "idx_pre_begin": idx_pre_begin,
    "idx_pre_end": idx_pre_end,
    "idx_stim_begin": idx_stim_begin,
    "idx_stim_end": idx_stim_end,
    "idx_post_begin": idx_post_begin,
    "idx_post_end": idx_post_end,
    "n_frames": n_frames
}

# save to mat file
if save_results:
    output_path = f"{output_folder}\\review_stim_data.mat"
    savemat(output_path, workspace_dict)